# 🎬 קולנוע המשנה — שוט SH09 עם תמונת ייחוס (ניסיון אחד)

זוהי **מחברת חדשה**, בנפרד ממחברת SDXL הישנה שנכשלה בהרכב הסצנה. היא משתמשת במודל **SDXL + ControlNet Canny + Image-to-Image**, כדי לנסות לשמור על שני האחים ועל בית האבן מתוך תמונת ייחוס קיימת.

**פעולות מינימליות:**
1. ב־Colab בחר **Runtime → Change runtime type → GPU (T4)**, אם עדיין אינו מופעל.
2. לחץ **Runtime → Run all**.
3. כשתופיע בקשה, בחר **תמונת ייחוס אחת** שיש בה כבר שני אחים ליד בית אבן בלילה. רצוי להשתמש בתמונת הייחוס הקולנועית שיצרנו יחד, ולא בתמונה הכושלת של החנות.
4. תוצאת הניסיון היחיד, אם תצליח, תוצג כקובץ ZIP להורדה. **אין הפעלה אוטומטית חוזרת**, אין יצירת 16 תמונות, ואין אישור אמנותי אוטומטי.

**עלות ופרטיות:** ההרצה נועדה ל־Colab Free GPU, כשזמין; Google עשויה להגביל או להפסיק הקצאת GPU. אם החשבון שלך משתמש במשאבים בתשלום, אל תפעיל עד שתבדוק את חיובו. התוכנה לא קוראת לשירות תמונות בתשלום ולא מפעילה overages; מכסת GPU חינמית אינה ניתנת לאימות כאן. תמונת הייחוס עולה **רק ל־runtime הפרטי שלך ב־Colab (שרת Google)**, לא ל־GitHub; אין לפרסם אותה אוטומטית.

**חשוב:** עדיין לא נבדקה בפועל ריצת GPU של המסלול החדש; מחברת זו היא ניסוי מבוקר אחד, לא הבטחה לתמונה מושלמת. אין שינוי באישורי נוסח, זכויות, פרסום או הלכה.


In [ ]:
# @title 1. בדיקת GPU לפני הורדות או שימוש במשאבים
import torch
if not torch.cuda.is_available():
    raise RuntimeError("לא נמצא GPU. בחר Runtime → Change runtime type → GPU (T4), ואל תמשיך על CPU.")
GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"GPU: {GPU_NAME}, VRAM: {VRAM_GB:.1f} GB")
if VRAM_GB < 12:
    raise RuntimeError("אין מספיק VRAM לפי בדיקת סף שמרנית (12GB). לא נעשה ניסיון GPU.")
print("בדיקת חומרה עברה; זו לא הבטחה למכסת Colab או להצלחת המודל.")


In [ ]:
# @title 2. בחירת תמונת ייחוס אחת (לא מועלית ל-GitHub)
import io
from pathlib import Path
from PIL import Image, ImageOps
from google.colab import files
print("בחר תמונת ייחוס קולנועית אחת: שני אחים ליד בית אבן בלילה.")
uploaded = files.upload()
if len(uploaded) != 1:
    raise RuntimeError("יש לבחור קובץ תמונה אחד בלבד.")
name, data = next(iter(uploaded.items()))
with Image.open(io.BytesIO(data)) as check:
    check.verify()
with Image.open(io.BytesIO(data)) as opened:
    picture = ImageOps.exif_transpose(opened).convert("RGB")
    if min(picture.size) < 512:
        raise RuntimeError("תמונת הייחוס קטנה מדי. צריך לפחות 512 פיקסלים בכל ממד.")
    PRIVATE_REFERENCE = Path("/content/mishnah-sh09-private-reference.png")
    picture.save(PRIVATE_REFERENCE, format="PNG")
print(f"תמונת ייחוס נטענה לזיכרון Colab בלבד: {picture.width}×{picture.height}")


In [ ]:
# @title 3. הורדת קוד מעודכן והתקנת ספריות ללא שירות בתשלום
import subprocess, sys
from pathlib import Path
REPO_DIR = Path("/content/mishnah-project")
REPO_URL = "https://github.com/barkinam21-pixel/project-mishna.git"
if REPO_DIR.exists():
    if not (REPO_DIR / ".git").exists():
        raise RuntimeError("יש תיקייה קיימת באותו שם שאינה מאגר Git; אין למחוק אותה אוטומטית.")
    subprocess.check_call(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", "main"])
    subprocess.check_call(["git", "-C", str(REPO_DIR), "checkout", "--detach", "FETCH_HEAD"])
else:
    subprocess.check_call(["git", "clone", "--quiet", "--depth", "1", "--branch", "main",
                           REPO_URL, str(REPO_DIR)])
if not (REPO_DIR / "tools/reference_guided.py").is_file():
    raise RuntimeError("קוד המסלול החדש עדיין לא קיים ב-main. אין להריץ את המסלול הישן.")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "diffusers>=0.35,<1", "transformers>=4.50,<5",
                       "accelerate>=1.2,<2", "safetensors>=0.5",
                       "opencv-python-headless>=4.9,<5", "Pillow>=10"])
print("הספריות הותקנו ב-Colab. לא נרכש GPU או שירות API.")


In [ ]:
# @title 4. בדיקה לפני ניסיון יחיד (ללא שימוש במודל)
import json, subprocess, sys
check = subprocess.run(
    [sys.executable, "tools/reference_guided.py", "preflight", "--reference",
     str(PRIVATE_REFERENCE), "--allow-generate"],
    cwd=str(REPO_DIR), capture_output=True, text=True)
if check.returncode:
    print(check.stdout, check.stderr)
    raise RuntimeError("בדיקת מוכנות טכנית נכשלה; לא נעשה ניסיון תמונה.")
readiness = json.loads(check.stdout)
print(json.dumps(readiness, indent=2, ensure_ascii=False))
if not readiness.get("can_generate"):
    raise RuntimeError("המסלול חסום בבדיקת מוכנות; לא נצרך ניסיון יצירה.")
print("בדיקות קדם-הפעלה עברו; תבוצע לכל היותר יצירת תמונה אחת.")


In [ ]:
# @title 5. ניסיון SH09 אחד בלבד, ביקורת אוטומטית ו-ZIP
import subprocess, sys
from pathlib import Path
from google.colab import files
command = [sys.executable, "tools/reference_guided.py", "all", "--reference",
           str(PRIVATE_REFERENCE), "--allow-generate"]
print("מריץ ניסיון אחד בלבד. אין יצירת אצווה ואין נסיונות חוזרים.", flush=True)
result = subprocess.run(command, cwd=str(REPO_DIR), text=True, check=False)
OUTPUT = REPO_DIR / "private-media/reference-guided/SH09"
archives = sorted(OUTPUT.glob("SH09-reference-guided-unapproved.zip"))
if result.returncode != 0 or not archives:
    raise RuntimeError("לא נוצר ZIP. בדוק את תוצאת הריצה מעל; נעילת ניסיון עשויה למנוע הרצה חוזרת.")
print("נוצר ZIP של איור טיוטה לא-מאושר. איכות ודיוק הסצנה טרם נבדקו בידי אדם.")
print("קובץ:", archives[-1])
files.download(str(archives[-1]))
